# FIDB PoC walkthrough: worker path and LanguageID sensitivity

This notebook follows one request through the real FIDB PoC, then reuses one
verified object from that run to answer a focused routing question:

**If the compiled bytes stay fixed and only Ghidra's LanguageID changes, do
the resulting Function ID hashes still match?**

Part 1 executes and verifies the normal worker. Part 2 continues with a
diagnostic raw-hash export that isolates the Ghidra analysis-model variable.
The notebook calls project APIs and checks their evidence; it does not
reimplement source retrieval, compilation, Ghidra analysis or FID equality.

## Before running

From the repository root, install the optional notebook group and start
Jupyter with the project environment and a Ghidra path valid in that shell:

```sh
uv sync --locked --group notebook
GHIDRA_HEADLESS=/opt/ghidra/support/analyzeHeadless uv run --group notebook jupyter lab notebooks/demo.ipynb
```

Inside Toolbx, use `/run/host/opt/ghidra/support/analyzeHeadless` instead and
set `JDK_JAVA_OPTIONS=-XX:-UseContainerSupport` if Java discovery requires it.
The notebook runs a real zlib build and LanguageID probe, recreating `work/`
and `output/`. Remove those generated directories after the demonstration
when preparing a source-only handoff.


## Part 1 Produce a candidate FIDB

### 1.1 Load the project

`fidb_poc.cli` is the worker entry point. `run_probe` is a separate diagnostic
API exposed by the same package. Ghidra remains responsible for disassembly
and Function ID computation.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from fidb_poc.language_probe import ProbeCase, run_probe


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find the checkout from either the repository root or notebooks/."""
    for candidate in (start.resolve(), *start.resolve().parents):
        pyproject = candidate / "pyproject.toml"
        if pyproject.is_file() and 'name = "fidb-poc"' in pyproject.read_text():
            return candidate
    raise RuntimeError("Run this notebook from the FIDB-POC checkout")


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


PROJECT_ROOT = find_project_root()
PROJECT_ROOT.name


### 1.2 Define the request

The request is **zlib + Linux x86-64 + native GNU GCC + the smoke profile**.
That short request crosses these existing project boundaries:

1. `fidb_poc.cli` parses the request.
2. `fidb_poc.config` resolves it to a checked-in recipe, route and treatment.
   A missing recipe is queued rather than guessed.
3. `fidb_poc.pipeline` retrieves the pinned source archive, verifies SHA-256,
   extracts it, and verifies the recipe's expected project/build markers.
4. `fidb_poc.adapters` returns the recipe's reviewed fixed build adapter; the
   worker validates compiler identity and the analysis artifacts' format and
   architecture. Link-phase treatments additionally validate linked output.
5. The pipeline gives Ghidra the route's explicit LanguageID and Ghidra
   compiler specification, generates a candidate `.fidb`, and records the
   result in `output/fidb_manifest.csv`.

The notebook supplies parameters only; source URLs, hashes, compiler commands
and flags remain declarative project data.


In [ ]:
REQUEST = {
    "library": "zlib",
    "version": "1.3.1",
    "route": "linux-x86_64-gnu-gcc",
    "treatment": "baseline_o2",
    "profile": "smoke",
}

CASES = (
    ProbeCase("default", "x86:LE:64:default"),
    ProbeCase("compat32", "x86:LE:64:compat32"),
)
COMPILER_SPEC = "gcc"  # Ghidra compiler specification, not the host compiler.
PROBE_OUTPUT = PROJECT_ROOT / "output/language_probe/notebook-x86-demo"


### 1.3 Run and verify the worker

This cell always enters through the public CLI. It does **not** use `--fresh`,
so a hash-verified downloaded archive may be reused. Every real run still
replaces worker-managed sources, builds, logs, Ghidra state, FIDBs and the
manifest. After the run, the cell selects the exact completed row and verifies
the downloaded source, every analysis object, and the candidate FIDB against
the checksums recorded by the worker.


In [ ]:
command = [
    sys.executable,
    "-m",
    "fidb_poc.cli",
    "--library",
    REQUEST["library"],
    "--route",
    REQUEST["route"],
    "--profile",
    REQUEST["profile"],
]
completed = subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout.replace(str(PROJECT_ROOT), "$PROJECT_ROOT").strip())

worker_manifest_path = PROJECT_ROOT / "output/fidb_manifest.csv"
with worker_manifest_path.open(newline="", encoding="utf-8") as stream:
    worker_rows = list(csv.DictReader(stream))

matches = [
    row
    for row in worker_rows
    if row["library"] == REQUEST["library"]
    and row["version"] == REQUEST["version"]
    and row["route"] == REQUEST["route"]
    and row["treatment"] == REQUEST["treatment"]
]
assert len(matches) == 1, f"Expected one manifest row, observed {len(matches)}"
worker_row = matches[0]
assert worker_row["status"] == "complete", worker_row["error"]

source_archive = (
    PROJECT_ROOT
    / "work/downloads"
    / f'{REQUEST["library"]}-{REQUEST["version"]}.tar.gz'
)
assert sha256(source_archive) == worker_row["source_sha256"]

artifact_paths = [
    PROJECT_ROOT / value
    for value in worker_row["analysis_artifact_path"].split(";")
    if value
]
artifact_hashes = [
    value for value in worker_row["analysis_artifact_sha256"].split(";") if value
]
assert len(artifact_paths) == len(artifact_hashes) == int(worker_row["object_count"])
for path, expected_hash in zip(artifact_paths, artifact_hashes):
    assert path.is_file()
    assert sha256(path) == expected_hash

fidb_path = PROJECT_ROOT / worker_row["fidb_path"]
assert fidb_path.is_file()
assert sha256(fidb_path) == worker_row["fidb_sha256"]

BINARY = next(path for path in artifact_paths if path.name == "adler32.o")
binary_sha256 = sha256(BINARY)

display(
    Markdown(
        "\n".join(
            [
                "**Verified worker result**",
                "",
                f'- Route: `{worker_row["route"]}`',
                f'- Compiler: `{worker_row["compiler_version"].split(" | ")[0]}`',
                f'- Source SHA-256: `{worker_row["source_sha256"]}`',
                f'- Analysis objects: `{worker_row["object_count"]}`',
                f'- FIDB: `{worker_row["fidb_path"]}`',
                f'- FIDB SHA-256: `{worker_row["fidb_sha256"]}`',
                f'- Status: `{worker_row["status"]}`',
            ]
        )
    )
)


## Part 2 Test LanguageID sensitivity

### 2.1 Freeze the diagnostic input

The normal worker has finished: it acquired, built, validated and recorded the
selected route. The diagnostic now reuses one exact object from that verified
result. Its SHA-256 is the boundary contract between the two parts.

This diagnostic does **not** query the generated `.fidb`. It exports raw Ghidra
FID hashes from the fixed object under each forced LanguageID. That narrower
design isolates the analysis-model variable.


In [ ]:
input_sha256 = sha256(BINARY)
assert input_sha256 == binary_sha256

display(
    Markdown(
        f"**Input:** `{BINARY.relative_to(PROJECT_ROOT)}`  \n"
        f"**Bytes:** `{BINARY.stat().st_size:,}`  \n"
        f"**SHA-256:** `{input_sha256}`  \n"
        f"**Cases:** `{', '.join(case.language_id for case in CASES)}`"
    )
)


### 2.2 Run the probe

For each case, `run_probe` creates an isolated Ghidra project, imports the same
bytes, forces `-processor` to the declared LanguageID, and holds the Ghidra
compiler specification fixed. Ghidra calculates Function IDs; the project
exporter records each hashable function's recovered entry point, full hash and
specific hash.

`fresh=True` replaces only this named diagnostic run. It does not remove the
worker's downloads, builds, manifest or FIDBs.


In [ ]:
probe_output = run_probe(
    project_root=PROJECT_ROOT,
    binary=BINARY,
    cases=CASES,
    compiler_spec=COMPILER_SPEC,
    output_directory=PROBE_OUTPUT,
    fresh=True,
)
print(probe_output.relative_to(PROJECT_ROOT))


### 2.3 Verify the result

The probe manifest records the fixed input, Ghidra version, LanguageIDs,
Ghidra compiler specification, script digests, and output checksums. This cell
verifies those checksums before reading the result.

Comparison identity is the **same recovered entry point**. Retention is the
percentage of the reference case's hashable functions whose full and specific
hashes are both unchanged in the comparison case. A reference entry point that
is missing in the comparison therefore reduces retention.


In [ ]:
probe_manifest_path = probe_output / "manifest.json"
probe_manifest = json.loads(probe_manifest_path.read_text(encoding="utf-8"))
assert probe_manifest["schema_version"] == "fidb-language-probe/v2"
assert probe_manifest["binary_sha256"] == input_sha256
assert probe_manifest["compiler_spec"] == COMPILER_SPEC

for output in probe_manifest["outputs"].values():
    output_path = probe_output / output["path"]
    assert sha256(output_path) == output["sha256"]

with (probe_output / "compatibility_matrix.csv").open(
    newline="", encoding="utf-8"
) as stream:
    matrix_rows = list(csv.DictReader(stream))

summary_rows = [
    {
        "reference": row["reference_case"],
        "comparison": row["comparison_case"],
        "shared": int(row["shared_entrypoints"]),
        "dual equal": int(row["dual_hash_equal"]),
        "reference retained": float(row["reference_retained_pct"]),
    }
    for row in matrix_rows
]

summary_rows


### 2.4 Interpret the result

The diagonal cells below compare each exported dataset with itself. They are
identity checks and are 100% by construction—not independent replay tests.
The off-diagonal cells contain the LanguageID comparison. Each label shows
equal/reference count and percentage.


In [ ]:
labels = [case.label for case in CASES]
rows_by_pair = {
    (row["reference_case"], row["comparison_case"]): row
    for row in matrix_rows
}
values = [
    [
        float(rows_by_pair[(reference, comparison)]["reference_retained_pct"])
        for comparison in labels
    ]
    for reference in labels
]

fig, ax = plt.subplots(figsize=(6.5, 5.2))
image = ax.imshow(values, vmin=0, vmax=100, cmap="RdYlGn")
ax.set_xticks(range(len(labels)), labels=labels)
ax.set_yticks(range(len(labels)), labels=labels)
ax.set_xlabel("Comparison case")
ax.set_ylabel("Reference case")
ax.set_title("Exact dual-hash retention by Ghidra LanguageID")
for row_index, reference in enumerate(labels):
    for column_index, comparison in enumerate(labels):
        row = rows_by_pair[(reference, comparison)]
        ax.text(
            column_index,
            row_index,
            f'{row["dual_hash_equal"]}/{row["reference_hashable"]}\n'
            f'{float(row["reference_retained_pct"]):.1f}%',
            ha="center",
            va="center",
            fontweight="bold",
        )
fig.colorbar(image, ax=ax, label="Reference functions retained (%)")
fig.tight_layout()
figure_path = probe_output / "compatibility_matrix.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()

off_diagonal = rows_by_pair[("default", "compat32")]
reference_count = int(off_diagonal["reference_hashable"])
comparison_count = int(off_diagonal["comparison_hashable"])
shared_count = int(off_diagonal["shared_entrypoints"])
equal_count = int(off_diagonal["dual_hash_equal"])
display(
    Markdown(
        f"**Result.** The two cases recovered {reference_count} and "
        f"{comparison_count} hashable functions; {shared_count} entry points "
        f"were shared, and {equal_count} retained the same full-and-specific "
        "hash pair. "
        "This demonstrates that LanguageID can alter exact FID matching; one "
        "object and a deliberately forced comparison make this a workflow "
        "demonstration, not a compatibility estimate."
    )
)


### Limitations and extension

This result controls LanguageID within one recorded local Ghidra installation.
It does not establish cross-version Ghidra reproducibility or transfer to an
untested route. A real PowerPC study would keep one verified PowerPC artifact
fixed and replace `CASES` with the relevant explicit PowerPC LanguageIDs (for
example `default`, `e500` and `4xx`), using the same probe and evidence checks.
